# vesicletrack — walkthrough

Detect, track and score vesicles in a time-lapse movie.

Run top to bottom. It works on a synthetic movie with **known ground truth**, so
you can see what a correct result looks like before pointing it at your own data.
Swap the path in section 2 when you are ready.

## 1. Setup

If the package is not installed (`pip install -e .`), the cell below adds `src/`
to the path so the notebook works from a fresh clone.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
root = root if (root / 'src').exists() else root.parent
sys.path.insert(0, str(root / 'src'))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from vesicletrack import Config, analyse, analyse_many, metrics, render
print('vesicletrack ready, working from', root)

## 2. Make (or point at) a movie

The synthetic movie has 40 static vesicles, 6 movers travelling 14 px, and 3 px of
stage drift. Replace `MOVIE` with your own `.tif` / `.nd2` when ready.

The stack must be **(T, Y, X)**. If yours has a channel or z axis, pass
`channel=` or `z_project='max'` to `analyse` — it will not guess which axis is time.

In [ ]:
import subprocess
MOVIE = root / 'examples' / 'synthetic.tif'
if not MOVIE.exists():
    subprocess.run([sys.executable, str(root/'examples'/'make_synthetic.py'),
                    str(MOVIE)], check=True)
print(MOVIE)

## 3. Parameters

Everything the pipeline does lives in the config. Two fields **must** be right for
your data and cannot be inferred from the images:

- `dt_seconds` — every rate scales through it
- `um_per_px` — leave `None` to stay in pixels

Set `dt_seconds` wrong and every rate is wrong while nothing *looks* wrong.

In [ ]:
cfg = Config.load(root / 'config' / 'default.yaml')

# synthetic movie: 20 fps, and it is shorter than the real recordings,
# so the coarse-graining windows are scaled down to match.
cfg.set('dt_seconds', 0.05)
cfg.set('um_per_px', 0.107)
cfg.set('metrics.tau_frames', 10)            # ~0.5 s
cfg.set('metrics.tau_directed_frames', 40)   # ~2 s
cfg.set('link.min_length_frames', 30)
cfg.set('output_dir', str(root / 'output'))   # absolute, so it lands here wherever the notebook is run from
cfg.validate()

print(f'tau        = {cfg.tau_seconds:.2f} s')
print(f'tau_direct = {cfg.tau_directed_seconds:.2f} s')

## 4. Run

load → drift-correct → detect → link → score → classify.

Drift correction runs **first**: stage drift moves every vesicle together, so
uncorrected it reads as directed transport in all of them at once.

In [ ]:
res = analyse(MOVIE, cfg, name='demo')
res.counts()

### Check the invariant

`gross ≥ directed ≥ net_coarse` is guaranteed by the triangle inequality for every
vesicle. If this is not empty, something upstream is wrong — duplicated frames,
unsorted tracks, NaNs — not the metric.

In [ ]:
bad = res.check()
print('ordering violations:', len(bad))
bad.head()

## 5. Look at the result

The three panels answer the two questions worth asking before trusting a number:
did detection **find** the vesicles, and did classification **label** them sensibly.

In [ ]:
out = res.save()
for k, v in out.items():
    print(f'{k:16s} {v}')

In [ ]:
from IPython.display import Image, display
display(Image(str(out['three_panel'])))

### The three distances

For the synthetic data the truth is known: the movers travelled 14 px ≈ 1.5 µm.
Watch what **gross** reports for the same vesicles.

In [ ]:
v = res.vesicles
cols = ['net_um','directed_um','gross_um','directed_p']
v.groupby('klass')[cols].median().round(3)

In [ ]:
display(Image(str(out['distances'])))

A single vesicle, with all three distances drawn on the same axes — the quickest
way to see why they differ so much.

In [ ]:
mv = res.movers().iloc[0]
p = root / 'output' / 'demo' / 'vesicles' / f"vesicle_{int(mv.particle):04d}.png"
display(Image(str(p))) if p.exists() else print('enable render.per_vesicle_images')

## 6. Tuning

`cfg.copy(**overrides)` makes a modified config without touching the original, so
a sweep is a loop rather than a series of edits.

Detection threshold is usually the first thing to set: too low invents spots, too
high loses dim vesicles.

In [ ]:
rows = []
for thr in [2.5, 3.0, 3.5, 4.0]:
    r = analyse(MOVIE, cfg.copy(**{'detect.threshold_sigma': thr}),
                name=f'thr{thr}', verbose=False)
    c = r.counts()
    rows.append(dict(threshold=thr, vesicles=r.summary['n_vesicles'],
                     movers=int(c.get('mover', 0)),
                     confined=int(c.get('confined', 0)),
                     excluded=int(c.get('excluded', 0))))
pd.DataFrame(rows)

Truth is 6 movers out of 46 vesicles. A threshold that changes the **mover** count
a lot is changing your answer, not just your detection sensitivity — worth knowing
before picking one.

### Does τ matter?

τ sets what counts as "consistent direction". Too small and localisation noise
survives; too large and genuine reversals are erased.

In [ ]:
rows = []
for tau in [10, 20, 40, 80]:
    r = analyse(MOVIE, cfg.copy(**{'metrics.tau_directed_frames': tau}),
                name=f'tau{tau}', verbose=False)
    g = r.vesicles.groupby('klass').directed.median()
    rows.append(dict(tau_frames=tau, tau_s=tau*cfg.dt_seconds,
                     mover=g.get('mover', np.nan),
                     confined=g.get('confined', np.nan)))
d = pd.DataFrame(rows)
d['separation'] = d.mover / d.confined
d.round(2)

The ratio is the signal-to-noise of the metric: how far the movers sit above the
confined population. It should rise with τ and then flatten.

## 7. Videos

Off by default — per-vesicle videos are one file each. `ffmpeg` is found on `PATH`
or next to the running Python; if it is missing you get a warning, not a crash.

In [ ]:
res_v = analyse(MOVIE, cfg.copy(**{'render.per_vesicle_videos': True,
                                  'render.overview_video': True,
                                  'render.max_vesicle_outputs': 3}),
                name='demo_video', verbose=False)
for k, v in res_v.save().items():
    print(f'{k:16s} {v}')

## 8. Batch

`analyse_many` returns one summary row per movie and keeps going if one fails —
an unreadable file should not cost the whole run.

```python
import glob
files = sorted(glob.glob('data/**/*.tif', recursive=True))
summary = analyse_many(files, cfg, output_dir='output')
summary.to_csv('output/batch_summary.csv', index=False)
```

Then pool the per-vesicle tables for statistics across conditions:

```python
frames = []
for f in sorted(pathlib.Path('output').glob('*/vesicles.csv')):
    frames.append(pd.read_csv(f).assign(movie=f.parent.name))
allv = pd.concat(frames, ignore_index=True)
```

**Aggregate per movie, not per vesicle.** Vesicles within one cell are not
independent; treating each as a replicate inflates n by a factor of hundreds and
will produce significance that does not survive a properly nested test.

## 9. Reproducibility

Every run writes `config_used.yaml` next to its outputs, so any figure can be
traced back to the exact parameters that produced it.

In [ ]:
print((root / 'output' / 'demo' / 'config_used.yaml').read_text()[:400])